In [109]:
import requests
import mysql.connector
from dotenv import load_dotenv
import os
import time
from decimal import Decimal

In [110]:
load_dotenv()

db = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME"),
    auth_plugin="caching_sha2_password"
)
cursor = db.cursor()

In [111]:
super_bowl_years = [year for year in range(2024, 1966 - 1, -1)]
teams = {'crd': 'Cardinals', 'atl': 'Falcons', 'rav': 'Ravens', 'buf': 'Bills', 'car': 'Panthers', 'chi': 'Bears', 'cin': 'Bengals', 'cle': 'Browns', 'dal': 'Cowboys', 'den': 'Broncos', 'det': 'Lions', 'gnb': 'Packers', 'htx': 'Texans', 'clt': 'Colts', 'jax': 'Jaguars', 'kan': 'Chiefs', 'rai': 'Raiders', 'sdg': 'Chargers', 'ram': 'Rams', 'mia': 'Dolphins', 'min': 'Vikings', 'nwe': 'Patriots', 'nor': 'Saints', 'nyg': 'Giants', 'nyj': 'Jets', 'phi': 'Eagles', 'pit': 'Steelers', 'sfo': '49ers', 'sea': 'Seahawks', 'tam': 'Buccaneers', 'oti': 'Titans', 'was': 'Commanders'}
season_ids = {2024: 1, 2023: 2, 2022: 3, 2021: 4, 2020: 5, 2019: 6, 2018: 7, 2017: 8, 2016: 9, 2015: 10, 2014: 11, 2013: 12, 2012: 13, 2011: 14, 2010: 15, 2009: 16, 2008: 17, 2007: 18, 2006: 19, 2005: 20, 2004: 21, 2003: 22, 2002: 23, 2001: 24, 2000: 25, 1999: 26, 1998: 27, 1997: 28, 1996: 29, 1995: 30, 1994: 31, 1993: 32, 1992: 33, 1991: 34, 1990: 35, 1989: 36, 1988: 37, 1987: 38, 1986: 39, 1985: 40, 1984: 41, 1983: 42, 1982: 43, 1981: 44, 1980: 45, 1979: 46, 1978: 47, 1977: 48, 1976: 49, 1975: 50, 1974: 51, 1973: 52, 1972: 53, 1971: 54, 1970: 55, 1969: 56, 1968: 57, 1967: 58, 1966: 59}
team_abr_ids = {'crd': 1, 'atl': 2, 'rav': 3, 'buf': 4, 'car': 5, 'chi': 6, 'cin': 7, 'cle': 8, 'dal': 9, 'den': 10, 'det': 11, 'gnb': 12, 'htx': 13, 'clt': 14, 'jax': 15, 'kan': 16, 'rai': 17, 'sdg': 18, 'ram': 19, 'mia': 20, 'min': 21, 'nwe': 22, 'nor': 23, 'nyg': 24, 'nyj': 25, 'phi': 26, 'pit': 27, 'sfo': 28, 'sea': 29, 'tam': 30, 'oti': 31, 'was': 32}
team_name_ids = {'Cardinals': 1, 'Falcons': 2, 'Ravens': 3, 'Bills': 4, 'Panthers': 5, 'Bears': 6, 'Bengals': 7, 'Browns': 8, 'Cowboys': 9, 'Broncos': 10, 'Lions': 11, 'Packers': 12, 'Texans': 13, 'Colts': 14, 'Jaguars': 15, 'Chiefs': 16, 'Raiders': 17, 'Chargers': 18, 'Rams': 19, 'Dolphins': 20, 'Vikings': 21, 'Patriots': 22, 'Saints': 23, 'Giants': 24, 'Jets': 25, 'Eagles': 26, 'Steelers': 27, '49ers': 28, 'Seahawks': 29, 'Buccaneers': 30, 'Titans': 31, 'Commanders': 32}

In [112]:
team_ids = [team_abr_ids[key] for key in team_abr_ids]
print(team_ids)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32]


In [113]:
data_per_game = {}
cursor.execute("""select * from games where season_id <= 55 order by season_id asc, week asc;""")
rows = cursor.fetchall()
for row in rows:
    data_per_game[row[0]] = []
print(len(data_per_game))

13397


In [114]:
for team_id in team_ids:
    cursor.execute("""select * from games where season_id <= 55 and (home_team_id = %s or away_team_id = %s) order by season_id desc, week asc;""", (team_id, team_id))
    rows = cursor.fetchall()
    team_game_stats = {}
    for i, row in enumerate(rows):
        stat_id = 0
        if row[2] == team_id:
            stat_id = row[4]
        else:
            stat_id = row[5]
        cursor.execute("""select * from game_stats where id = %s;""", (stat_id, ))
        result = cursor.fetchall()
        if len(result) != 1:
            print("ERRRROR")
        team_game_stats[row[0]] = [float(item) if isinstance(item, Decimal) else item for item in result[0]]

    team_game_stats_adjusted = {}
    for key in team_game_stats:
        team_game_stats_adjusted[key] = []
        masking = [1, 1, 1, 1, 1]
        for i, stat in enumerate(team_game_stats[key]):
            if stat == None:
                team_game_stats_adjusted[key].append(-1)
                masking[len(team_game_stats[key]) - i - 1] = 0
            elif i >= 3 and i != 7 and i != 4:
                team_game_stats_adjusted[key].append(stat)
            
        team_game_stats_adjusted[key].extend(masking)

    curr5 = []
    curr10 = []
    curr25 = []

    for i, key in enumerate(team_game_stats_adjusted):

        if i < 5:
            curr5.append(team_game_stats_adjusted[key])
        else:
            mean = np.mean(curr5, axis=0).tolist()
            avg_5 = [round(float(num), 3) for num in mean]
            avg_5[-5:] = [0 if fault < 1 else fault for fault in avg_5[-5:]]
            avg_5.pop(5)
            avg_5.pop(6)
            curr5.pop(0)
            curr5.append(team_game_stats_adjusted[key])

        if i < 10:
            curr10.append(team_game_stats_adjusted[key])
        else:
            last_10_games = curr10.copy()
            mean = np.mean(curr10, axis=0).tolist()
            avg_10 = [round(float(num), 3) for num in mean]
            avg_10[-5:] = [0 if fault < 1 else fault for fault in avg_5[-5:]]
            avg_10.pop(5)
            avg_10.pop(6)
            curr10.pop(0)
            curr10.append(team_game_stats_adjusted[key])

        if i < 25:
            curr25.append(team_game_stats_adjusted[key])
        else:
            mean = np.mean(curr25, axis=0).tolist()
            avg_25 = [round(float(num), 3) for num in mean]
            avg_25[-5:] = [0 if fault < 1 else fault for fault in avg_5[-5:]]
            avg_25.pop(5)
            avg_25.pop(6)
            curr25.pop(0)
            curr25.append(team_game_stats_adjusted[key])

            #Enter data when there are 25 past games
            data_per_game[key].append({'Team': team_game_stats[key][2], 'Last 10': last_10_games, 'Avg 5': avg_5, 'Avg 10': avg_10, 'Avg 25': avg_25})

In [115]:
data_per_game = {k: v for k, v in data_per_game.items() if len(v) == 2}

print(len(data_per_game))


12919


In [116]:
cursor.execute("""select * from games where season_id <= 55 and season_id >= 14 order by season_id desc, week asc;""")
rows = cursor.fetchall()
games_in_order = []
for row in rows:
    if row[0] in data_per_game:
        games_in_order.append(row[0])
print(games_in_order)

[16202, 16086, 16099, 16112, 16126, 16139, 16151, 16175, 16192, 16207, 16214, 16223, 16245, 16258, 16087, 16100, 16113, 16127, 16140, 16152, 16166, 16176, 16193, 16215, 16224, 16235, 16259, 16088, 16101, 16114, 16128, 16141, 16167, 16177, 16184, 16208, 16216, 16236, 16246, 16250, 16142, 16153, 16194, 16260, 16154, 16225, 16155, 15886, 15900, 15914, 15940, 15953, 15964, 15978, 16008, 16016, 16023, 16028, 16044, 16061, 15887, 15901, 15915, 15928, 15941, 15954, 15965, 15979, 15989, 15999, 16009, 16035, 16045, 15888, 15902, 15916, 15929, 15942, 15966, 15980, 16017, 16036, 16050, 16055, 16062, 16067, 15889, 15903, 15917, 15930, 15943, 15955, 15967, 16000, 16018, 16029, 16037, 16056, 16070, 15890, 15904, 15918, 15931, 15944, 15968, 15981, 15990, 16024, 16030, 16051, 16063, 16073, 15891, 15905, 15919, 15932, 15945, 15956, 15969, 15982, 16001, 16010, 16025, 16052, 16057, 15892, 15906, 15920, 15946, 15957, 15970, 15991, 16002, 16011, 16019, 16053, 16058, 16064, 15893, 15907, 15921, 15933, 15947

In [117]:
training_data = []
training_winners = []

for game in games_in_order:
    team1_id = data_per_game[game][0]['Team']
    game_data = []

    game_data.append(team1_id)
    game_data.append(data_per_game[game][1]['Team'])
    for x in [0,1]:
        for i in data_per_game[game][x]['Last 10']:
            for j in i:
                game_data.append(j)
    for x in [0,1]:
        for i in data_per_game[game][x]['Avg 5']:
            game_data.append(i)
    for x in [0,1]:
        for i in data_per_game[game][x]['Avg 10']:
            game_data.append(i)
    for x in [0,1]:
        for i in data_per_game[game][x]['Avg 25']:
            game_data.append(i)

    cursor.execute("""select * from game_stats where game_id = %s and team_id = %s;""", (game, team1_id))
    training_winners.append(cursor.fetchall()[0][4])
    training_data.append(game_data)

In [118]:
cursor.execute("""select * from games where season_id < 14 order by season_id desc, week asc;""")
rows = cursor.fetchall()
games_in_order = []
for row in rows:
    if row[0] in data_per_game:
        games_in_order.append(row[0])
print(games_in_order)
print(1 in games_in_order)

[3391, 3407, 3424, 3444, 3459, 3473, 3502, 3513, 3523, 3534, 3546, 3559, 3579, 3593, 3626, 3631, 3392, 3408, 3425, 3445, 3460, 3474, 3487, 3514, 3535, 3560, 3570, 3594, 3599, 3604, 3637, 3644, 3393, 3409, 3426, 3446, 3461, 3475, 3488, 3515, 3524, 3536, 3547, 3571, 3586, 3595, 3614, 3621, 3394, 3410, 3427, 3447, 3476, 3489, 3525, 3537, 3548, 3561, 3587, 3605, 3638, 3645, 3657, 3395, 3411, 3428, 3448, 3462, 3477, 3490, 3503, 3526, 3549, 3562, 3600, 3622, 3648, 3396, 3412, 3429, 3491, 3527, 3538, 3550, 3572, 3588, 3606, 3623, 3627, 3639, 3652, 3397, 3430, 3449, 3463, 3478, 3492, 3504, 3551, 3580, 3628, 3632, 3640, 3654, 3398, 3413, 3464, 3505, 3516, 3528, 3539, 3552, 3573, 3589, 3607, 3615, 3624, 3653, 3399, 3414, 3431, 3450, 3465, 3479, 3493, 3540, 3574, 3590, 3596, 3625, 3633, 3641, 3415, 3432, 3451, 3466, 3480, 3494, 3517, 3541, 3575, 3591, 3601, 3608, 3616, 3646, 3400, 3433, 3452, 3467, 3481, 3495, 3506, 3529, 3542, 3563, 3576, 3597, 3609, 3649, 3401, 3416, 3434, 3453, 3468, 3482, 349

In [119]:
testing_data = []
testing_winners = []

for game in games_in_order:
    team1_id = data_per_game[game][0]['Team']
    game_data = []

    game_data.append(team1_id)
    game_data.append(data_per_game[game][1]['Team'])
    for x in [0,1]:
        for i in data_per_game[game][x]['Last 10']:
            for j in i:
                game_data.append(j)
    for x in [0,1]:
        for i in data_per_game[game][x]['Avg 5']:
            game_data.append(i)
    for x in [0,1]:
        for i in data_per_game[game][x]['Avg 10']:
            game_data.append(i)
    for x in [0,1]:
        for i in data_per_game[game][x]['Avg 25']:
            game_data.append(i)

    cursor.execute("""select * from game_stats where game_id = %s and team_id = %s;""", (game, team1_id))
    testing_winners.append(cursor.fetchall()[0][4])
    testing_data.append(game_data)

In [104]:
print(len(training_data))
print(len(testing_data))

for i in testing_data:
    if (len(i) != 926):
        print(i)

9378
3541


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
embedding_dim = 8  # Dimension for team_id embeddings
hidden_dim = 128  # Number of LSTM hidden units
num_layers = 2    # Number of LSTM layers
batch_size = 64   # Batch size
num_epochs = 50   # Number of epochs
learning_rate = 0.001

In [ ]:
training_data = torch.tensor(training_data)
training_winners = torch.tensor(training_winners)




train_team_ids = training_data[:, :2].long()
train_stats = training_data[:, 2:]
test_team_ids = testing_data[:, :2].long()
test_stats = testing_data[:, 2:]

In [121]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Hyperparameters
embedding_dim = 8  # Dimension for team_id embeddings
hidden_dim = 128  # Number of LSTM hidden units
num_layers = 2    # Number of LSTM layers
batch_size = 64   # Batch size
num_epochs = 50   # Number of epochs
learning_rate = 0.001

# Extract team IDs and stats
train_team_ids = training_data[:, :2].long()
train_stats = training_data[:, 2:]
test_team_ids = testing_data[:, :2].long()
test_stats = testing_data[:, 2:]

# Dataset and DataLoader
train_dataset = TensorDataset(train_team_ids, train_stats, training_winners)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = TensorDataset(test_team_ids, test_stats, testing_winners)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Define the model
class LSTMModel(nn.Module):
    def __init__(self, num_teams, embedding_dim, input_dim, hidden_dim, num_layers):
        super(LSTMModel, self).__init__()
        self.embedding = nn.Embedding(num_teams, embedding_dim)
        self.lstm = nn.LSTM(input_dim + 2 * embedding_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, team_ids, stats):
        # Embed the team IDs
        team_embeds = self.embedding(team_ids)
        team_embeds = team_embeds.view(team_embeds.size(0), -1)  # Flatten the embeddings

        # Concatenate embeddings and stats
        x = torch.cat((team_embeds, stats), dim=1)
        x = x.unsqueeze(1)  # Add sequence dimension for LSTM

        # Pass through LSTM
        lstm_out, _ = self.lstm(x)
        lstm_out = lstm_out[:, -1, :]  # Use the last output of the sequence

        # Fully connected layer
        out = self.fc(lstm_out)
        out = self.sigmoid(out).squeeze()
        return out

# Initialize the model
num_teams = torch.max(torch.cat((train_team_ids, test_team_ids))) + 1
model = LSTMModel(num_teams=num_teams, embedding_dim=embedding_dim, input_dim=924, hidden_dim=hidden_dim, num_layers=num_layers)

# Loss and optimizer
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Training loop
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for team_ids, stats, winners in train_loader:
        optimizer.zero_grad()
        outputs = model(team_ids, stats)
        loss = criterion(outputs, winners.float())
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")

# Testing loop
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for team_ids, stats, winners in test_loader:
        outputs = model(team_ids, stats)
        predicted = (outputs > 0.5).int()
        total += winners.size(0)
        print(winners)
        correct += (predicted == winners).sum().item()

accuracy = correct / total
print(f"Test Accuracy: {accuracy:.4f}")

# Save the model
torch.save(model.state_dict(), "nfl_lstm_model.pth")


TypeError: list indices must be integers or slices, not tuple

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

# Hyperparameters
embedding_dim = 8  # Dimension for team_id embeddings
hidden_dim = 128  # Number of LSTM hidden units
num_layers = 2    # Number of LSTM layers
batch_size = 64   # Batch size
num_epochs = 50   # Number of epochs
learning_rate = 0.001

# Extract team IDs and stats
train_team_ids = training_data[:, :2].long()
train_stats = training_data[:, 2:]
test_team_ids = testing_data[:, :2].long()
test_stats = testing_data[:, 2:]

# Standardize the stats
scaler = StandardScaler()
train_stats = scaler.fit_transform(train_stats.numpy())  # Fit to train stats and transform
test_stats = scaler.transform(test_stats.numpy())  # Transform test stats using the same scaler

# Convert back to tensors
train_stats = torch.tensor(train_stats, dtype=torch.float32)
test_stats = torch.tensor(test_stats, dtype=torch.float32)

# Dataset and DataLoader
train_dataset = TensorDataset(train_team_ids, train_stats, training_winners)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = TensorDataset(test_team_ids, test_stats, testing_winners)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Define the model
class LSTMModel(nn.Module):
    def __init__(self, num_teams, embedding_dim, input_dim, hidden_dim, num_layers):
        super(LSTMModel, self).__init__()
        self.embedding = nn.Embedding(num_teams, embedding_dim)
        self.lstm = nn.LSTM(input_dim + 2 * embedding_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, team_ids, stats):
        # Embed the team IDs
        team_embeds = self.embedding(team_ids)
        team_embeds = team_embeds.view(team_embeds.size(0), -1)  # Flatten the embeddings

        # Concatenate embeddings and stats
        x = torch.cat((team_embeds, stats), dim=1)
        x = x.unsqueeze(1)  # Add sequence dimension for LSTM

        # Pass through LSTM
        lstm_out, _ = self.lstm(x)
        lstm_out = lstm_out[:, -1, :]  # Use the last output of the sequence

        # Fully connected layer
        out = self.fc(lstm_out)
        out = self.sigmoid(out).squeeze()
        return out

# Initialize the model
num_teams = torch.max(torch.cat((train_team_ids, test_team_ids))) + 1
model = LSTMModel(num_teams=num_teams, embedding_dim=embedding_dim, input_dim=924, hidden_dim=hidden_dim, num_layers=num_layers)

# Loss and optimizer
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Training loop
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for team_ids, stats, winners in train_loader:
        optimizer.zero_grad()
        outputs = model(team_ids, stats)
        loss = criterion(outputs, winners.float())
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")

# Testing loop
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for team_ids, stats, winners in test_loader:
        outputs = model(team_ids, stats)
        predicted = (outputs > 0.5).int()
        total += winners.size(0)
        correct += (predicted == winners).sum().item()

accuracy = correct / total
print(f"Test Accuracy: {accuracy:.4f}")

# Save the model
torch.save(model.state_dict(), "nfl_lstm_model.pth")
